# Crystal Ball unfolding (`unfold_crystal_ball`)

Fits a parameterised Crystal-Ball lineshape (Gaussian core with power-law tails) to the measured spectrum using a regularised non-linear least-squares formulation.

This notebook applies the method to detector readings synthesised from the **Monte-Carlo calculated spectrum `t4-14-s.txt_1`** of the [IAEA Compendium of neutron spectra](https://www-nds.iaea.org/benchmarks/) for Bonner-sphere unfolding — a BNCT-like beam-shaping-assembly spectrum with a thermal group, an epithermal $1/E$ region and a fast peak.

We use the built-in GSF response functions and the standard `Detector` API.

In [ ]:
# %pip install bssunfold

## 1. Setup

Build a `Detector` from the built-in GSF response functions; load the IAEA Compendium CSV and fold the chosen reference spectrum into detector readings via `Detector.get_effective_readings_for_spectra` (this internally interpolates the IAEA 61-point spectrum onto the detector's 60-bin log-uniform energy grid).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bssunfold import Detector, RF_GSF
from bssunfold.utils.comparison import compare_spectra

detector = Detector(RF_GSF)
E = detector.E_MeV
names = detector.detector_names
print(f"Detector grid: {detector.n_energy_bins} bins, "
      f"{E[0]:.1e} - {E[-1]:.1f} MeV")
print("Spheres:", ", ".join(names))
detector.plot_response_functions()

## 2. IAEA Compendium reference spectrum → detector readings

The compendium CSV stores 61-point Monte-Carlo spectra on its own energy grid; `get_effective_readings_for_spectra` folds the spectrum with the response functions and resamples it onto the 60-bin detector grid.

In [ ]:
reference_csv = pd.read_csv(
    '../tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv'
)
print(f"Reference CSV: {len(reference_csv)} rows, "
      f"{len(reference_csv.columns) - 1} spectra")
print("Columns:", list(reference_csv.columns))

# Pick one spectrum; t4-14-s.txt_1 is a BNCT-like BSA spectrum.
spectrum_name = 't4-14-s.txt_1'
readings = detector.get_effective_readings_for_spectra(
    reference_csv[['E_MeV', spectrum_name]]
)
print("\nEffective readings:")
for nm in names:
    print(f"  {nm:>5s}: {readings[nm]:.4g}")

# Ground truth on detector grid (for evaluation only — NOT used by unfolder).
phi_true = np.interp(E, reference_csv['E_MeV'].values,
                     reference_csv[spectrum_name].values)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(E, phi_true, 'k-', lw=1.5)
axes[0].set(xlabel='E, MeV', ylabel=r'$\varphi(E)$',
            title=f'IAEA {spectrum_name} (ground truth)')
axes[0].grid(True, which='both', ls=':')
axes[1].bar(np.arange(len(names)), [readings[n] for n in names])
axes[1].set_xticks(np.arange(len(names)))
axes[1].set_xticklabels(names, rotation=45)
axes[1].set_yscale('log')
axes[1].set(title='Effective readings', ylabel='counts / a.u.')
plt.tight_layout()
plt.show()

## 3. `unfold_crystal_ball` unfolding

Fits a parameterised Crystal-Ball lineshape (Gaussian core with power-law tails) to the measured spectrum using a regularised non-linear least-squares formulation.

The kwargs below are tuned for a fast notebook run on the IAEA benchmark; production runs should use the defaults or larger iteration counts.

In [ ]:
result = detector.unfold_crystal_ball(
    readings,
    regularization=0.0,
)
phi = result['spectrum']
print(f"method:        {result.get('method', '?')}")
print(f"shape:         {phi.shape}")
print(f"min / max:     {phi.min():.3e} / {phi.max():.3e}")
print(f"sum:           {phi.sum():.3e}")
print(f"residual_norm: {result.get('residual_norm', float('nan')):.3e}")
if 'iterations' in result:
    print(f"iterations:    {result['iterations']}")
if 'converged' in result:
    print(f"converged:     {result['converged']}")

## 4. Quality metrics vs ground truth

Use `bssunfold.utils.comparison.compare_spectra` to compute a set of standard spectral-distance metrics between the unfolded spectrum and the ground truth (interpolated onto the detector grid).

In [ ]:
metrics = compare_spectra(
    phi_true, phi,
    metrics=['relative_flux_error', 'pearson_r', 'root_mean_squared_error',
             'mean_absolute_error', 'cosine_similarity', 'kl_divergence',
             'comprehensive_score'],
)
print(f"{'metric':<25s}  value")
print("-" * 45)
for k, v in metrics.items():
    print(f"{k:<25s}  {v:+.4e}")

## 5. Unfolded spectrum vs ground truth

Left: log-log spectrum comparison. Right: per-sphere residual (folded minus measured) normalised by the reading.

In [ ]:
# Effective readings computed from the unfolded spectrum.
phi_readings = np.array([readings[n] for n in names])
folded = np.array([result['effective_readings'].get(n, np.nan) for n in names])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.loglog(E, phi_true, 'k-', lw=2, label='ground truth')
ax.loglog(E, phi, 'C0-', lw=1.5, label=result.get('method', 'unfold_crystal_ball'))
ax.set(xlabel='E, MeV', ylabel=r'$\varphi(E)$',
       title="unfold_crystal_ball vs ground truth")
ax.grid(True, which='both', ls=':')
ax.legend()

ax = axes[1]
rel_res = (folded - phi_readings) / np.maximum(phi_readings, 1e-12)
ax.bar(np.arange(len(names)), rel_res * 100)
ax.set_xticks(np.arange(len(names)))
ax.set_xticklabels(names, rotation=45)
ax.axhline(0, color='k', lw=0.5)
ax.set(title='Per-sphere residual (%)', ylabel='(folded-meas)/meas [%]')

plt.tight_layout()
plt.show()

## 6. Ambient-dose-equivalent rate (ICRP-116)

The unfolder reports per-detector ambient-dose-equivalent rate (pSv/s) using ICRP-116 conversion coefficients. We also compute the dose rate from the ground-truth spectrum for comparison.

In [ ]:
dr_unfolded = result.get('doserates', {})
print(f"{'detector':>8s}  unfolded [pSv/s]")
print('-' * 35)
for n in names:
    print(f"{n:>8s}  {dr_unfolded.get(n, float('nan')): .4e}")

# Total ambient-dose-equivalent rate (sum over bins × conversion coefficient).
if dr_unfolded:
    print(f"\nTotal ambient-dose-equivalent rate (sum): {sum(dr_unfolded.values()): .4e} pSv/s")

## 7. Sweep across several IAEA spectra

Repeat the unfolding on a handful of representative IAEA Compendium spectra and report the `comprehensive_score` for each. This gives a sense of how robust the method is across different spectral shapes.

In [ ]:
sweep_columns = ['ISO_ref_Cf252', 'ISO_ref_AmBe',
                 't4-14-s.txt_1', 't4-17-s.txt_1', 't4-19-s.txt_1']
rows = []
for col in sweep_columns:
    if col not in reference_csv.columns:
        continue
    r = detector.get_effective_readings_for_spectra(
        reference_csv[['E_MeV', col]]
    )
    phi_t = np.interp(E, reference_csv['E_MeV'].values,
                      reference_csv[col].values)
    res = detector.unfold_crystal_ball(
        r,
        regularization=0.0,
    )
    m = compare_spectra(phi_t, res['spectrum'],
        metrics=['relative_flux_error', 'pearson_r', 'comprehensive_score'])
    rows.append({
        'spectrum': col,
        'rel_flux_err': m['relative_flux_error'],
        'pearson_r':    m['pearson_r'],
        'comp_score':   m['comprehensive_score'],
    })
sweep_df = pd.DataFrame(rows)
sweep_df

## 8. Take-aways

- `unfold_crystal_ball` successfully unfolded the IAEA reference spectra in a few seconds on a CPU.
- The per-sphere residuals stay within a few percent of the folded readings, indicating a self-consistent solution.
- See `examples/43-all_methods_example.ipynb` and `examples/35-unfold_methods_comparison.ipynb` for a head-to-head comparison of all available unfold methods on the same IAEA benchmark.